# Reproduce eagle's performance cards on Kaggle

This notebook only calls the scripts in `eagle/benchmarks/reproduce/`
(see its README). Before running:

- **Settings → Accelerator**: GPU T4 x2 (the card uses one of the two GPUs).
- **Settings → Internet**: on (the bootstrap downloads the pinned toolchain and wheels).

A Kaggle session ends after 12 hours. The run is split into resumable steps and
stops cleanly before the deadline. To continue in a new session: save this
version (its `/kaggle/working/cards` becomes the notebook output), add that output
as an input of the next session, set `RESUME_FROM` to its path and run again.

In [ ]:
# Parameters
PRESET = "smoke"        # smoke (minutes) | short | full (the committed cards' matrix)
WHICH = "gpu"           # gpu | cpu | all  (cpu/all also build the CPU environment)
EAGLE_REF = "v0.4.0"    # eagle ref to fetch the scripts from; the other repos use
REFS = {}               # their release tags unless set here, e.g. {"hawk": "main"}
SESSION_HOURS = 11.5    # stop starting new steps after this long (Kaggle limit: 12 h)
RESUME_FROM = ""        # e.g. "/kaggle/input/<this-notebook>/cards" to continue a run

In [ ]:
import os, pathlib, shutil, subprocess, time
T0 = time.time()
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU, as the cards measure one device
WORK = pathlib.Path("/kaggle/temp" if pathlib.Path("/kaggle/temp").is_dir() else "/tmp") / "raptor-repro"
SRC, OUT = WORK / "src", pathlib.Path("/kaggle/working/cards")
SRC.mkdir(parents=True, exist_ok=True)
if not (SRC / "eagle").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", EAGLE_REF,
                    "https://github.com/amasat01/eagle.git", str(SRC / "eagle")], check=True)
REPRO = SRC / "eagle" / "benchmarks" / "reproduce"
REF_ARGS = " ".join(f"--ref {k}={v}" for k, v in REFS.items())
print(REPRO)

In [ ]:
# Bootstrap: pinned toolchain + wheels, the four repositories, eagle's _core for this GPU.
profiles = {"gpu": ["gpu"], "cpu": ["cpu"], "all": ["gpu", "cpu"]}[WHICH]
for prof in profiles:
    subprocess.run(["bash", str(REPRO / "bootstrap.sh"), "--target", "kaggle", "--profile", prof,
                    "--work", str(WORK), "--src", str(SRC), *REF_ARGS.split()], check=True)

In [ ]:
# Resume: bring back the steps and passes a previous session saved.
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUT, dirs_exist_ok=True)
    print("resumed from", RESUME_FROM)

In [ ]:
# Run the cards, split into resumable steps; stops before the session deadline (exit 3).
left = max(0.1, SESSION_HOURS - (time.time() - T0) / 3600)
rc = subprocess.run(["bash", str(REPRO / "run_cards.sh"), "--work", str(WORK), "--which", WHICH,
                     "--preset", PRESET, "--out", str(OUT), "--split",
                     "--deadline-h", f"{left:.2f}"]).returncode
print({0: "all cards done", 3: "deadline reached: save this version and resume in a new session"}
      .get(rc, f"failed (exit {rc}): see the output above"))

In [ ]:
# The cards written so far (Markdown tables rendered from the JSON).
from IPython.display import Markdown, display
for p in sorted(OUT.glob("*/*_card/*.md")):
    display(Markdown(f"### {p.parent.parent.name} / {p.name}\n\n" + p.read_text()))